# Predict GOLD Stain from DAPI
The pretrained pix2pix architecture is further trained using the WGAN-GP Cross-Zamirski model to predict the GOLD stain from cropped DAPI Nuclei Images. The discriminator is a CNN without any normalization.
This uses preset hyperparameters
Here, we optimize this model.

In [ ]:
import copy
import pathlib
import random
import shutil
import sys

import albumentations as A
import joblib
import mlflow
import numpy as np
import optuna
import pandas as pd
import torch

## Find the root of the git repo on the host system

In [ ]:
# Get the current working directory
cwd = pathlib.Path.cwd()

if (cwd / ".git").is_dir():
    root_dir = cwd

else:
    root_dir = None
    for parent in cwd.parents:
        if (parent / ".git").is_dir():
            root_dir = parent
            break

# Check if a Git root directory was found
if root_dir is None:
    raise FileNotFoundError("No Git root directory found.")

## Custom Imports

In [ ]:
sys.path.append(str((root_dir / "1.develop_vision_models").resolve(strict=True)))
sys.path.append(str((root_dir / "1.develop_vision_models/losses").resolve(strict=True)))
sys.path.append(str((root_dir / "1.develop_vision_models/models").resolve(strict=True)))
sys.path.append(str((root_dir / "1.develop_vision_models/utils").resolve(strict=True)))

from CombinedBatchNormCPLoss import CombinedBatchNormCPLoss
from datasets.ImageMetaDataset import ImageMetaDataset
from Pix2PixDiscriminatorNoNorm import Pix2PixDiscriminator
from trainers.WGANGPCPPix2PixMetaTrainer import WGANGPCPPix2PixMetaTrainer
from transforms.MinMaxNormalize import MinMaxNormalize
from unet_model import UNet
from WassersteinCPTextureVarianceGeneratorLoss import \
    WassersteinCPTextureVarianceGeneratorLoss
from WassersteinGradientPenaltyLoss import WassersteinGradientPenaltyLoss

## Set random seeds

In [ ]:
random.seed(0)
np.random.seed(0)
torch.manual_seed(0)

mlflow.log_param("random_seed", 0)

# Inputs

In [ ]:
conda_env = str(root_dir / "1.develop_vision_models" / "environment.yml")

# Nuclei crops path of treated nuclei in the Dapi channel with all original pixel values
treated_dapi_crops = (
    root_dir
    / "vision_nuclear_speckle_prediction/treated_nuclei_dapi_crops_same_background"
).resolve(strict=True)

# Nuclei crops path of nuclei in the Gold channel with all original pixel values
gold_crops = (
    root_dir / "vision_nuclear_speckle_prediction/gold_cropped_nuclei_same_background"
).resolve(strict=True)

# Paths to original nuclear speckle data
data_dir = (root_dir / "nuclear_speckles_data").resolve(strict=True)
nuclear_mask_dir = (data_dir / "Nuclear_masks").resolve(strict=True)
sc_profiles_path = list(
    (data_dir / "Preprocessed_data/single_cell_profiles")
    .resolve(strict=True)
    .glob("*feature_selected*.parquet")
)

# Load single-cell profile data
scdfs = [pd.read_parquet(sc_path) for sc_path in sc_profiles_path if sc_path.is_file()]
scdfs = pd.concat(scdfs, axis=0).reset_index(drop=True)

pipeline_path = root_dir / "1.develop_vision_models/utils/sc_crop_analysis.cppipe"

# Also hard-coded in WGANGPPix2PixMetaTrainer
img_montage_dir = pathlib.Path("generated_image_epoch_montage")

# Outputs

In [ ]:
figure_path = pathlib.Path("pix2pix_validation_images")
figure_path.mkdir(parents=True, exist_ok=True)

metrics_path = pathlib.Path("metrics")
metrics_path.mkdir(parents=True, exist_ok=True)

model_path = pathlib.Path("model")
model_path.mkdir(parents=True, exist_ok=True)

In [ ]:
description = """
- Architecture: Cross-Zamirski CP (Pix2pix wgangp CP) without conditional component
- Image Modification: Cropped to nuclei using CP bounding box and min-max normalized
- No normalization layers in the discriminator model
- Batch normalization is present in the generator model
- Pretrained Generator model (from best Cross-Zamirski model)
- Additional Loss Modification: Includes the following CP loss features:
    - Texture_SumVariance_sc_crop_3_00_256
    - Texture_SumVariance_sc_crop_3_02_256
    - Texture_SumVariance_sc_crop_3_01_256
    - Texture_SumVariance_sc_crop_3_03_256
    - Texture_Variance_sc_crop_3_00_256
    - Texture_Variance_sc_crop_3_02_256
    - Texture_Variance_sc_crop_3_01_256
    - Texture_Variance_sc_crop_3_03_256
    - Granularity_1_sc_crop
    - Texture_Contrast_sc_crop_3_02_256
    - Texture_Contrast_sc_crop_3_01_256
    - Texture_Contrast_sc_crop_3_03_256
    - Texture_Contrast_sc_crop_3_00_256
    - Intensity_TotalIntensity_sc_crop
    - Granularity_2_sc_crop
    - Granularity_3_sc_crop
    - Granularity_4_sc_crop
    - Granularity_5_sc_crop
    - Granularity_6_sc_crop
    - Granularity_7_sc_crop
    - ImageQuality_PowerLogLogSlope_sc_crop
- The CP loss component is weighted by a hyperparameter importance factor (+lambda*CP)
"""

mlflow.set_tag("mlflow.note.content", description)

# Initialize and Train Model

In [ ]:
input_transforms = A.Compose(
    [
        MinMaxNormalize(_normalization_factor=(2**16) - 1, _always_apply=True),
    ]
)

target_transforms = copy.deepcopy(input_transforms)

img_dataset = ImageMetaDataset(
    _input_dir=treated_dapi_crops,
    _target_dir=gold_crops,
    _input_transform=input_transforms,
    _target_transform=target_transforms,
)

In [ ]:
model_data = {
    "best_loss": float("inf"),
}
hyperparams_idx = 0
lr_generator = 0.0013981961408994052
lr_discriminator = 0.0006431172050131992

mlflow.set_tag("model_architecture", "pix2pix")

mlflow.log_param("optimizer_generator", "adam")
mlflow.log_param("optimizer_discriminator", "adam")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

trainer_params = {"epochs": 100, "patience": 25, "example_images_per_epoch": 10}

optimizer_params = {}
model_hyperparams = {}

discriminator_model = Pix2PixDiscriminator(
    _number_input_channels=1, _number_output_channels=32, _conv_depth=4
)
generator_model = mlflow.pytorch.load_model(
    "file:///home/camo/projects/nuclear_speckles_analysis/mlruns/598485576074396081/e6ee9d58e9554e18b475bd3f617cd011/artifacts/generator_model"
)

discriminator_model.to(device)
generator_model.to(device)

trainer_params["batch_size"] = 17
trainer_params["discriminator_update_frequency"] = 1

optimizer_params["lr_generator"] = lr_generator
optimizer_params["lr_discriminator"] = lr_discriminator

model_hyperparams["wasserstein_gp_importance"] = 1

model_hyperparams["reconstruction_importance"] = 1

model_hyperparams["cp_loss_importance"] = 1

mlflow.log_params(trainer_params)
trainer_params = {
    f"_{param_name}": param_value
    for param_name, param_value in trainer_params.items()
}
trainer_params["_save_pretrained_generated_imgs"] = True

optimizers = {
    "generator_optimizer": torch.optim.Adam(
        generator_model.parameters(),
        lr=optimizer_params["lr_generator"],
        betas=(0.5, 0.999),
    ),
    "discriminator_optimizer": torch.optim.Adam(
        discriminator_model.parameters(),
        lr=optimizer_params["lr_discriminator"],
        betas=(0.5, 0.999),
    ),
}

for opt_name, opt in optimizers.items():
    opt_params = opt.param_groups[0].copy()

    del opt_params["params"]
    opt_params = {
        f"{opt_name}_{opt_param_name}": opt_param
        for opt_param_name, opt_param in opt_params.items()
    }
    opt_params[opt_name] = opt.__class__.__name__.lower()

    mlflow.log_params(opt_params)

mlflow.log_params(trainer_params)
mlflow.log_params(model_hyperparams)

cp_texture_var_loss = CombinedBatchNormCPLoss(
    _metric_name="cp_combined_batch_norm_loss",
    _pipeline_path=pipeline_path,
    _targets_path=gold_crops,
    _feature_names=[
        "Texture_SumVariance_sc_crop_3_00_256",
        "Texture_SumVariance_sc_crop_3_02_256",
        "Texture_SumVariance_sc_crop_3_01_256",
        "Texture_SumVariance_sc_crop_3_03_256",
        "Texture_Variance_sc_crop_3_00_256",
        "Texture_Variance_sc_crop_3_02_256",
        "Texture_Variance_sc_crop_3_01_256",
        "Texture_Variance_sc_crop_3_03_256",
        "Granularity_1_sc_crop",
        "Texture_Contrast_sc_crop_3_02_256",
        "Texture_Contrast_sc_crop_3_01_256",
        "Texture_Contrast_sc_crop_3_03_256",
        "Texture_Contrast_sc_crop_3_00_256",
        "Intensity_TotalIntensity_sc_crop",
        "Granularity_2_sc_crop",
        "Granularity_3_sc_crop",
        "Granularity_4_sc_crop",
        "Granularity_5_sc_crop",
        "Granularity_6_sc_crop",
        "Granularity_7_sc_crop",
        "ImageQuality_PowerLogLogSlope_sc_crop",
    ],
)

trainer = WGANGPCPPix2PixMetaTrainer(
    _generator_model=generator_model,
    _discriminator_model=discriminator_model,
    _image_dataset=img_dataset,
    _generator_optimizer=optimizers["generator_optimizer"],
    _discriminator_optimizer=optimizers["discriminator_optimizer"],
    _discriminator_loss=WassersteinGradientPenaltyLoss(
        "discriminator_wgan_gp_cross_zamirski_classification_average",
        _gradient_penalty_importance=model_hyperparams[
            "wasserstein_gp_importance"
        ],
    ),
    _generator_loss=WassersteinCPTextureVarianceGeneratorLoss(
        _metric_name="generator_wgan_gp_cp_texture_variance_average",
        _cp_loss=cp_texture_var_loss,
        _cp_loss_importance=model_hyperparams["cp_loss_importance"],
        _reconstruction_importance=model_hyperparams[
            "reconstruction_importance"
        ],
    ),
    **trainer_params,
)

loss, best_generator, best_discriminator = trainer.train()

mlflow.pytorch.log_model(
    pytorch_model=best_generator.cpu(),
    artifact_path="generator_model",
    conda_env=conda_env,
)
mlflow.pytorch.log_model(
    pytorch_model=best_discriminator.cpu(),
    artifact_path="discriminator_model",
    conda_env=conda_env,
)

for img_set in img_montage_dir.iterdir():
    if img_set.is_dir():
        mlflow.log_artifacts(img_set, artifact_path=img_set.name)

shutil.rmtree(img_montage_dir)

In [ ]:
mlflow.log_artifact("pix2pix_optuna_study.joblib")